In [36]:
import json

with open("datasets/syn_sentence_list.json", "r") as f:
    artificial_data = json.load(f)

len(artificial_data)

15017

In [64]:
import pandas as pd
from sklearn.model_selection import train_test_split

test_size = 0.2
random_state = 0

tram_df = pd.read_json("datasets/tram_train.json")
df_train, df_val = train_test_split(tram_df, test_size=test_size, random_state=random_state)
df_train.shape, df_val.shape

((12286, 3), (3072, 3))

In [ ]:
import loader

model = loader.load_model_for_embedding("sentence-transformers/all-mpnet-base-v2")

In [65]:
from collections import Counter

# Flatten the list of labels and count the occurrences of each label
label_counts = Counter(label for labels in tram_df['labels'] for label in labels)
print(len(tram_df['labels']))
# Convert the counter to a DataFrame for better visualization
label_distribution = pd.DataFrame.from_dict(label_counts, orient='index', columns=['count']).sort_values(by='count', ascending=False)
print(label_distribution)

15358
           count
T1027        557
T1140        386
T1059.003    293
T1055        237
T1105        201
T1106        163
T1090        135
T1071.001    132
T1082        101
T1078         97
T1053.005     94
T1112         87
T1003.001     83
T1204.002     75
T1083         75
T1566.001     69
T1021.001     68
T1047         67
T1057         66
T1041         65
T1070.004     64
T1574.002     61
T1036.005     60
T1562.001     60
T1573.001     56
T1095         54
T1190         53
T1547.001     52
T1218.011     50
T1056.001     48
T1005         43
T1110         43
T1016         43
T1570         41
T1219         40
T1113         39
T1543.003     38
T1033         36
T1518.001     27
T1548.002     24
T1074.001     20
T1569.002     20
T1012         19
T1552.001     17
T1564.001     13
T1210         12
T1072         10
T1484.001     10
T1068          9
T1557.001      7


## Synthetic Data (Rebalanced)

In [12]:
import numpy as np
from tqdm import tqdm

alpha = 0.3
beta = 0.9

artificial_data_selected = []

for idx, row in tqdm(df_train.iterrows(), total=df_train.shape[0]):
    sentence = row['sentence']
    labels = row["labels"]
    sent_encoding = model.encode(sentence)
    artificial_sent_w_labels = artificial_data.get(sentence, [])
    augmented_sentences = [item['augmented_sentence'] for item in artificial_sent_w_labels]
    artificial_sent_encoding = model.encode(augmented_sentences)
    similarity = model.similarity(sent_encoding, artificial_sent_encoding)
    
    # Find indices where similarity is between alpha and beta
    indices = np.where((similarity >= alpha) & (similarity <= beta))[1]
    
    # if no sentence matches criteria, skip it
    if len(indices) < 1:
        continue

    # if sentence has no labels, then select a random sentence
    if len(labels) == 0:
        rand_idx = np.random.choice(indices)
        artificial_data_selected.append(artificial_sent_w_labels[rand_idx])
    else:
        for idx in indices:
            selected_sentence = artificial_sent_w_labels[idx]
            artificial_data_selected.append(selected_sentence)

100%|██████████| 12286/12286 [12:02<00:00, 17.01it/s]


In [13]:
from collections import Counter

# Get the index with the maximum similarity
# Extract labels from artificial_data_selected
selected_labels = [label for item in artificial_data_selected for label in item['labels']]

# Count the occurrences of each label
selected_label_counts = Counter(selected_labels)

# Convert the counter to a DataFrame for better visualization
selected_label_distribution = pd.DataFrame.from_dict(selected_label_counts, orient='index', columns=['count']).sort_values(by='count', ascending=False)
print(selected_label_distribution)

           count
T1027       2154
T1140       1392
T1059.003   1345
T1055        820
T1105        809
T1106        563
T1071.001    560
T1090        514
T1082        484
T1053.005    476
T1083        393
T1078        388
T1003.001    357
T1112        352
T1057        350
T1041        324
T1562.001    306
T1070.004    301
T1021.001    285
T1204.002    272
T1095        248
T1573.001    240
T1056.001    236
T1047        234
T1566.001    231
T1016        224
T1113        224
T1005        221
T1190        221
T1036.005    220
T1219        210
T1574.002    192
T1547.001    177
T1570        172
T1218.011    165
T1110        161
T1518.001    155
T1543.003    137
T1033        135
T1548.002     92
T1012         91
T1074.001     82
T1569.002     68
T1552.001     65
T1564.001     60
T1210         43
T1484.001     36
T1072         27
T1068         22
T1557.001     22


In [14]:
import random

artificial = {
    "sentence": [],
    "labels": []
}

for label in label_distribution.index:
    count = label_distribution.loc[label, 'count']
    # get from selected_artificial_data the sentences with the label
    selected_sentences = [item for item in artificial_data_selected if label in item['labels']]
    random_sentences = random.sample(selected_sentences, count)
    artificial['sentence'].extend([data['augmented_sentence'] for data in random_sentences])
    artificial['labels'].extend([data['labels'] for data in random_sentences])

for data in artificial_data_selected:
    if len(data['labels']) == 0:
        artificial['sentence'].append(data['augmented_sentence'])
        artificial['labels'].append(data['labels'])


In [15]:
artificial_df = pd.DataFrame(artificial)
artificial_df.drop_duplicates(subset=['sentence'], inplace=True)
artificial_df

,sentence,labels
0,COOLCLIENT's arsenal includes obfuscation tech...,[T1027]
1,The resolution of functions via hashing was si...,[T1027]
2,The loader is configured with multiple flags t...,[T1027]
3,Discard the PE headers that were implanted in ...,[T1027]
4,The decryption process using the key was unsuc...,[T1027]
...,...,...
13064,A .NET-based malware known as CUEMiner functio...,[]
13065,The infected master boot record initiates a tr...,[]
13066,The installation process of the service dictat...,[]
13067,The distinction between local-only and network...,[]


In [16]:
augmented_df = pd.concat([df_train, artificial_df], ignore_index=True)
augmented_df.reset_index(drop=True, inplace=True)
augmented_df

,sentence,labels,doc_title
0,"It quarantines malware, even if the process is...",[],Analyzing Solorigate the compromised DLL file ...
1,"As with HTTP/HTTPS traffic, DNS traffic can be...",[],Cobalt Strike a Defenders Guide Part 2
2,Royal ransomware’s attack flow Installation Ou...,[T1021.001],Conti Team One Splinter Group Resurfaces as Ro...
3,The function in the embedded DLL that decrypts...,[T1140],These arent the apps youre looking for fake in...
4,Oldest Linux sample gets uploaded to Virus Tot...,[],Iron Tigers SysUpdate Reappears Adds Linux Tar...
...,...,...,...
25051,A .NET-based malware known as CUEMiner functio...,[],NaN
25052,The infected master boot record initiates a tr...,[],NaN
25053,The installation process of the service dictat...,[],NaN
25054,The distinction between local-only and network...,[],NaN


In [17]:
augmented_df['doc_title'].fillna('artificial_data', inplace=True)

/tmp/ipykernel_1477490/2275702299.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  augmented_df['doc_title'].fillna('artificial_data', inplace=True)


In [18]:
# Shuffle the dataframe
augmented_df_shuffled = augmented_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

# Save to JSON file
augmented_df_shuffled.to_json("datasets/tram_train_augmented_artificial.json")